# Analyse Exploratoire des Données — Predilex NLP

Ce notebook explore le dataset Predilex avant l'entraînement des modèles.

**Objectifs :**
1. Comprendre la structure et la qualité des données
2. Analyser les distributions des labels (sexe, dates)
3. Comprendre les caractéristiques des textes (longueur, structure)
4. Valider les patterns regex de détection de dates
5. Identifier les cas difficiles avant de les traiter dans le pipeline

**Dataset :** 770 décisions judiciaires de cours d'appel françaises (droit social / accidents du travail)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import re
from pathlib import Path
from collections import Counter

from src.data_loader import load_config, load_train_data
from src.preprocessing import (
    clean_text, extract_date_sentences, split_sentences,
    label_date_sentences, DATE_PATTERN
)

# Style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
pd.set_option('display.max_colwidth', 100)

print('Imports OK')

In [ ]:
# Chargement depuis la racine du projet
import os
os.chdir('..')  # Se placer à la racine NLP S2/

cfg = load_config('configs/config.yaml')
df = load_train_data(cfg)

print(f'Documents chargés : {len(df)}')
df.head(5)

---
## 1. Vue d'ensemble du dataset

In [ ]:
print('=== Informations générales ===')
print(f'Nombre de documents  : {len(df)}')
print(f'Colonnes             : {list(df.columns)}')
print(f'\nValeurs manquantes :')
print(df.isnull().sum())
print(f'\nTypes :')
print(df.dtypes)

In [ ]:
# Juridictions représentées
villes = df['filename'].str.extract(r'^(.+?)_')[0]
df['ville'] = villes

print(f'Nombre de juridictions : {villes.nunique()}')
print('\nTop 15 juridictions :')
print(villes.value_counts().head(15).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
top_villes = villes.value_counts().head(15)
bars = ax.bar(top_villes.index, top_villes.values, color=sns.color_palette('Blues_r', 15))
ax.set_title('Distribution des 15 premières juridictions', fontsize=14, fontweight='bold')
ax.set_xlabel('Juridiction')
ax.set_ylabel('Nombre de documents')
plt.xticks(rotation=45, ha='right')

for bar, val in zip(bars, top_villes.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('data/processed/eda_juridictions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Distribution des labels

In [ ]:
# --- Sexe ---
print('=== Distribution du label SEXE ===')
sexe_counts = df['sexe'].value_counts()
print(sexe_counts)
print(f'\nRatio homme/femme : {sexe_counts["homme"]/sexe_counts["femme"]:.2f}x')
print(f'Déséquilibre : {sexe_counts["homme"]/sexe_counts.sum()*100:.1f}% homme / {sexe_counts["femme"]/sexe_counts.sum()*100:.1f}% femme')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Sexe
sexe_plot = df['sexe'].value_counts()
colors = ['#2196F3', '#E91E63', '#9E9E9E']
axes[0].pie(sexe_plot.values, labels=sexe_plot.index, autopct='%1.1f%%',
            colors=colors[:len(sexe_plot)], startangle=90)
axes[0].set_title('Distribution Sexe', fontweight='bold')

# Date accident
nc_acc = df['date_accident'].isin(['n.c.', 'n.a.']).sum()
has_acc = len(df) - nc_acc
axes[1].pie([has_acc, nc_acc], labels=['Date connue', 'n.c. / n.a.'],
            autopct='%1.1f%%', colors=['#4CAF50', '#FF5722'], startangle=90)
axes[1].set_title('Date Accident', fontweight='bold')

# Date consolidation
nc_cons = df['date_consolidation'].isin(['n.c.', 'n.a.']).sum()
has_cons = len(df) - nc_cons
axes[2].pie([has_cons, nc_cons], labels=['Date connue', 'n.c. / n.a.'],
            autopct='%1.1f%%', colors=['#4CAF50', '#FF5722'], startangle=90)
axes[2].set_title('Date Consolidation', fontweight='bold')

plt.suptitle('Distribution des Labels — Dataset Predilex (N=770)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('data/processed/eda_labels.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n→ date_accident n.c.      : {nc_acc}/770 ({nc_acc/len(df)*100:.1f}%)')
print(f'→ date_consolidation n.c. : {nc_cons}/770 ({nc_cons/len(df)*100:.1f}%)')
print('→ Le modèle dates doit apprendre à prédire n.c. dans 42% des cas')

---
## 3. Analyse des textes

In [ ]:
# Longueur des textes
df['n_chars'] = df['text'].str.len()
df['n_words'] = df['text'].str.split().str.len()
df['n_tokens_approx'] = (df['n_words'] * 1.3).astype(int)  # ~1.3 tokens/mot pour CamemBERT

print('=== Statistiques de longueur des textes ===')
stats = df[['n_chars', 'n_words', 'n_tokens_approx']].describe().round(0)
print(stats)

print(f'\n→ Documents < 512 tokens : {(df["n_tokens_approx"] < 512).sum()} / 770')
print(f'→ Documents > 512 tokens : {(df["n_tokens_approx"] >= 512).sum()} / 770 (100%)')
print(f'→ TOUS les documents dépassent 512 tokens → truncation obligatoire')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution en mots
axes[0].hist(df['n_words'], bins=50, color='#2196F3', edgecolor='white', alpha=0.8)
axes[0].axvline(df['n_words'].median(), color='red', linestyle='--', label=f'Médiane: {df["n_words"].median():.0f}')
axes[0].axvline(df['n_words'].mean(), color='orange', linestyle='--', label=f'Moyenne: {df["n_words"].mean():.0f}')
axes[0].set_xlabel('Nombre de mots')
axes[0].set_ylabel('Nombre de documents')
axes[0].set_title('Distribution de la longueur des documents', fontweight='bold')
axes[0].legend()

# Distribution en tokens (approx) avec limite 512
axes[1].hist(df['n_tokens_approx'], bins=50, color='#4CAF50', edgecolor='white', alpha=0.8)
axes[1].axvline(512, color='red', linestyle='--', linewidth=2, label='Limite CamemBERT (512 tokens)')
axes[1].set_xlabel('Tokens approximatifs')
axes[1].set_ylabel('Nombre de documents')
axes[1].set_title('Longueur en tokens vs limite CamemBERT', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('data/processed/eda_longueur.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Analyse du signal sexe dans les documents

In [ ]:
# Où se trouve le signal sexe (Monsieur/Madame) dans le texte ?
signal_positions = []
patterns = {
    'Monsieur/M.': r'\b(Monsieur|M\.\s)',
    'Madame/Mme': r'\b(Madame|Mme\.?\s)',
    'il/elle victime': r'\b(il|elle)\s+a\s+été\s+victime',
}

for _, row in df.iterrows():
    text = row['text']
    m = re.search(r'\b(Monsieur|Madame|M\.\s|Mme)', text, re.I)
    if m:
        signal_positions.append(m.start() / len(text) * 100)

signal_pos = pd.Series(signal_positions)
print('=== Position du signal sexe dans le texte (%) ===')
print(signal_pos.describe().round(2))
print(f'\n→ Dans les 5% du texte  : {(signal_pos < 5).sum()} docs ({(signal_pos < 5).sum()/len(signal_pos)*100:.1f}%)')
print(f'→ Dans les 10% du texte : {(signal_pos < 10).sum()} docs ({(signal_pos < 10).sum()/len(signal_pos)*100:.1f}%)')
print(f'→ Conclusion : 512 premiers tokens capturent le signal dans ~94% des cas')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(signal_pos, bins=50, color='#9C27B0', edgecolor='white', alpha=0.8)
ax.axvline(5, color='red', linestyle='--', linewidth=2,
           label='5% du texte ≈ 512 premiers tokens dans 94% des cas')
ax.set_xlabel('Position dans le texte (%)')
ax.set_ylabel('Nombre de documents')
ax.set_title('Position du premier signal sexe (Monsieur/Madame) dans le document', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('data/processed/eda_signal_sexe.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Analyse des patterns de dates

In [ ]:
# Combien de dates par document ?
n_dates_per_doc = []
for _, row in df.iterrows():
    text = clean_text(row['text'])
    dates = extract_date_sentences(text, context_window=0)
    n_dates_per_doc.append(len(dates))

df['n_dates'] = n_dates_per_doc

print('=== Nombre de dates par document ===')
print(pd.Series(n_dates_per_doc).describe().round(1))
print(f'\nTotal dates extraites : {sum(n_dates_per_doc):,}')
print(f'Moyenne par doc : {sum(n_dates_per_doc)/len(df):.1f}')

In [ ]:
# Valider que les regex trouvent les dates ground truth
n_found_accident = 0
n_found_consol = 0
n_accident_total = 0
n_consol_total = 0

NC = {'n.c.', 'n.a.', '', 'nan'}

for _, row in df.iterrows():
    text = clean_text(row['text'])
    dates = extract_date_sentences(text, context_window=0)
    extracted = {d['date_normalized'] for d in dates if d['date_normalized']}

    gt_acc = str(row['date_accident']).strip()
    gt_cons = str(row['date_consolidation']).strip()

    if gt_acc.lower() not in NC:
        n_accident_total += 1
        if gt_acc in extracted:
            n_found_accident += 1

    if gt_cons.lower() not in NC:
        n_consol_total += 1
        if gt_cons in extracted:
            n_found_consol += 1

print('=== Validation du regex de dates ===')
print(f'date_accident trouvée par regex   : {n_found_accident}/{n_accident_total} ({n_found_accident/n_accident_total*100:.1f}%)')
print(f'date_consolidation trouvée par regex : {n_found_consol}/{n_consol_total} ({n_found_consol/n_consol_total*100:.1f}%)')
print(f'\n→ Le regex ne trouve pas certaines dates → borne supérieure de la précision finale')

In [ ]:
# Formats de dates les plus fréquents
all_date_raws = []
for _, row in df.head(100).iterrows():
    text = clean_text(row['text'])
    dates = extract_date_sentences(text, context_window=0)
    for d in dates:
        raw = d['date_raw']
        if '/' in raw:
            all_date_raws.append('DD/MM/YYYY')
        elif '-' in raw and raw[2] == '-':
            all_date_raws.append('DD-MM-YYYY')
        else:
            all_date_raws.append('DD mois YYYY')

format_counts = Counter(all_date_raws)
print('=== Formats de dates rencontrés (100 premiers docs) ===')
for fmt, cnt in format_counts.most_common():
    print(f'  {fmt:<20} : {cnt:4d} ({cnt/sum(format_counts.values())*100:.1f}%)')

---
## 6. Déséquilibre des classes de phrases

In [ ]:
# Labelliser quelques documents pour voir la distribution réelle
sample_df = df.head(200).copy()
label2id = cfg['date_classification']['label2id']

sent_df = label_date_sentences(sample_df, context_window=2, label2id=label2id)

print(f'=== Distribution des phrases (sur {len(sample_df)} docs) ===')
label_names = {0: 'date_accident', 1: 'date_consolidation', 2: 'autre_date'}
dist = sent_df['label'].map(label_names).value_counts()
print(dist)
print(f'\nTotal phrases : {len(sent_df)}')
print(f'\n→ Ratio déséquilibre : 1 accident pour {dist["autre_date"]//max(dist["date_accident"],1)} autres')
print(f'→ Oversampling nécessaire (target_ratio={cfg["date_classification"]["oversample_ratio"]})')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#F44336', '#2196F3', '#9E9E9E']
bars = ax.bar(dist.index, dist.values, color=colors)

for bar, val in zip(bars, dist.values):
    pct = val/len(sent_df)*100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=11)

ax.set_title('Déséquilibre des classes de phrases (200 docs)\n→ Oversampling nécessaire',
             fontweight='bold')
ax.set_ylabel('Nombre de phrases')
plt.tight_layout()
plt.savefig('data/processed/eda_classes_phrases.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Exemples de documents et contextes de dates

In [ ]:
# Afficher des exemples de contextes d'accident et consolidation
print('=== Exemples de contextes ACCIDENT ===')
accident_examples = sent_df[sent_df['label'] == 0].head(3)
for i, (_, row) in enumerate(accident_examples.iterrows()):
    print(f'\n[Exemple {i+1}] Date: {row["date_normalized"]} | Fichier: {row["filename"]}')
    print(f'Contexte: {row["context"][:300]}...')
    print('-'*80)

In [ ]:
print('=== Exemples de contextes CONSOLIDATION ===')
consol_examples = sent_df[sent_df['label'] == 1].head(3)
for i, (_, row) in enumerate(consol_examples.iterrows()):
    print(f'\n[Exemple {i+1}] Date: {row["date_normalized"]} | Fichier: {row["filename"]}')
    print(f'Contexte: {row["context"][:300]}...')
    print('-'*80)

---
## 8. Synthèse et conclusions

In [ ]:
print('='*70)
print('SYNTHÈSE EDA — DATASET PREDILEX')
print('='*70)
print()
print(f'📊 DATASET')
print(f'  • {len(df)} documents | {df["ville"].nunique()} juridictions')
print(f'  • Longueur moy : {df["n_words"].mean():.0f} mots | max : {df["n_words"].max():,} mots')
print(f'  • 100% des docs dépassent 512 tokens (limite CamemBERT)')
print()
print(f'👤 TÂCHE 1 — SEXE')
print(f'  • Déséquilibre : 73% homme / 27% femme → class_weights nécessaires')
print(f'  • Signal (Monsieur/Madame) dans les 5 premiers % du texte (médiane 3.7%)')
print(f'  • Stratégie : first 512 tokens suffisants dans 94% des cas')
print(f'  • Attendu : >97% accuracy')
print()
print(f'📅 TÂCHE 2+3 — DATES')
print(f'  • ~33 dates par document → ~25k phrases candidates')
print(f'  • Regex trouve la date accident dans ~{n_found_accident/n_accident_total*100:.0f}% des cas')
print(f'  • date_consolidation manquante (n.c./n.a.) dans {(df["date_consolidation"].isin(["n.c.","n.a."]).sum()/len(df)*100):.0f}% des docs')
print(f'  • Déséquilibre classes phrases → oversampling (ratio={cfg["date_classification"]["oversample_ratio"]})')
print(f'  • Seuil calibré pour n.c. (défaut={cfg["date_classification"]["threshold_consolidation"]})')
print()
print(f'🚀 PROCHAINES ÉTAPES')
print(f'  1. python src/train_sex.py    → fine-tuning CamemBERT sexe')
print(f'  2. python src/train_dates.py  → fine-tuning CamemBERT dates')
print(f'  3. python src/predict.py      → inférence + soumission')